# Tutorial 3: Working with TENDL (TALYS-based Evaluated Nuclear Data Library)

## Overview

TENDL is a comprehensive nuclear data library that provides evaluated cross-sections for a wide range of isotopes. Unlike ENDF, which is manually evaluated, TENDL uses the TALYS nuclear reaction code to systematically generate evaluations.

### What you'll learn:
- What makes TENDL unique compared to other libraries
- How to download TENDL data files
- How to process TENDL data (similar format to ENDF)
- How to compare TENDL with ENDF evaluations
- Understanding the advantages and limitations of automated evaluations

### Prerequisites:
```bash
pip install matplotlib numpy pandas requests
```

## 1. Understanding TENDL

### Key Features:

- **Automated Evaluation**: Uses TALYS code for consistent evaluations
- **Extensive Coverage**: Data for ~2800 isotopes (vs ~400 in ENDF)
- **Regular Updates**: Annual releases (TENDL-2021, TENDL-2023, etc.)
- **Format**: Uses ENDF-6 format (same as ENDF/B)
- **Use Cases**: Medical isotopes, fusion, activation studies

### TENDL vs ENDF:

| Feature | TENDL | ENDF |
|---------|-------|------|
| Evaluation Method | Automated (TALYS) | Manual expert evaluation |
| Isotope Coverage | ~2800 isotopes | ~400 isotopes |
| Quality for Major Isotopes | Good | Excellent |
| Uncertainties | Provided | Limited |
| Update Frequency | Annual | Every 3-5 years |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import requests
import io

# Create data directory
data_dir = Path('../data/tendl')
data_dir.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

## 2. Accessing TENDL Data

TENDL data can be downloaded from: https://tendl.web.psi.ch/

The data is organized by:
- Release year (2017, 2019, 2021, 2023)
- Projectile type (neutron, proton, deuteron, etc.)
- Target isotope

Files are in ENDF-6 format, so we can use the same tools as Tutorial 1.

In [ ]:
# TENDL download information
TENDL_BASE_URL = "https://tendl.web.psi.ch/tendl_2021/neutron_file/"

def get_tendl_filename(element, mass):
    """
    Generate TENDL filename for a given isotope.
    
    Parameters:
    -----------
    element : str
        Element symbol (e.g., 'U', 'Fe')
    mass : int
        Mass number (e.g., 235, 238)
    
    Returns:
    --------
    str : TENDL filename
    """
    # TENDL uses format: n-Element-XXX.tendl
    # Example: n-U-235.tendl
    return f"n-{element}-{mass:03d}.tendl"

def download_tendl(element, mass, output_dir=data_dir):
    """
    Download TENDL data file for a specific isotope.
    
    Note: This is a template function. TENDL website may require
    registration or have access restrictions.
    """
    filename = get_tendl_filename(element, mass)
    url = TENDL_BASE_URL + filename
    output_path = output_dir / filename
    
    print(f"TENDL download information:")
    print(f"  Isotope: {element}-{mass}")
    print(f"  Filename: {filename}")
    print(f"  URL: {url}")
    print(f"\nNote: TENDL may require registration at https://tendl.web.psi.ch/")
    print(f"      For this tutorial, we'll work with simulated TENDL-like data.")
    
    return None

# Example: Try to get U-235 info
download_tendl('U', 235)

## 3. Processing TENDL Files

Since TENDL uses ENDF-6 format, we can use OpenMC (from Tutorial 1) to read the files.

For this tutorial, we'll simulate TENDL data based on TALYS calculations.

In [ ]:
def simulate_tendl_data(isotope='U-235', reaction='fission'):
    """
    Simulate TENDL-like cross-section data.
    
    In practice, you would read actual TENDL files using:
    import openmc.data
    data = openmc.data.IncidentNeutron.from_endf('n-U-235.tendl')
    """
    energy = np.logspace(-2, 7, 10000)  # eV
    
    if isotope == 'U-235' and reaction == 'fission':
        # Thermal region
        thermal = 582.0 * np.sqrt(0.0253 / np.maximum(energy, 1e-5))
        
        # Resonance region (TALYS produces smoother curves)
        resonance = 48 + 25 * np.sin(np.log(np.maximum(energy, 1)))
        
        # Fast region
        fast = 2.05 + 0.48 * np.log10(np.maximum(energy, 1e5) / 1e5)
        
        xs = np.where(energy < 1, thermal, np.where(energy < 1e4, resonance, fast))
        
    elif isotope == 'U-235' and reaction == 'capture':
        # (n,γ) capture cross-section
        thermal = 98.0 * np.sqrt(0.0253 / np.maximum(energy, 1e-5))
        resonance = 15 + 10 * np.sin(np.log(np.maximum(energy, 1)))
        fast = 0.5 + 0.1 * np.log10(np.maximum(energy, 1e5) / 1e5)
        
        xs = np.where(energy < 1, thermal, np.where(energy < 1e4, resonance, fast))
    
    else:
        xs = np.ones_like(energy)
    
    return energy, xs

# Generate TENDL data for U-235
energy_tendl, xs_fission_tendl = simulate_tendl_data('U-235', 'fission')
_, xs_capture_tendl = simulate_tendl_data('U-235', 'capture')

print("TENDL-like data generated")
print(f"Energy range: {energy_tendl.min():.2e} - {energy_tendl.max():.2e} eV")
print(f"Data points: {len(energy_tendl)}")

## 4. Visualizing TENDL Cross-Sections

Let's plot the TENDL evaluations for U-235.

In [ ]:
plt.figure(figsize=(14, 8))

plt.loglog(energy_tendl, xs_fission_tendl, label='Fission (MT=18)', linewidth=2, color='red')
plt.loglog(energy_tendl, xs_capture_tendl, label='Capture (MT=102)', linewidth=2, color='blue')
plt.loglog(energy_tendl, xs_fission_tendl + xs_capture_tendl, 
           label='Absorption (Fission + Capture)', linewidth=2, 
           color='green', linestyle='--')

plt.xlabel('Neutron Energy (eV)', fontsize=14)
plt.ylabel('Cross-section (barns)', fontsize=14)
plt.title('U-235 Neutron Cross-Sections (TENDL-2021)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3, which='both')
plt.xlim(1e-2, 1e7)
plt.tight_layout()
plt.savefig(data_dir / 'tendl_u235_cross_sections.png', dpi=150)
plt.show()

## 5. Comparing TENDL with ENDF

One of the most valuable uses of TENDL is to compare different evaluations. Let's compare TENDL and ENDF for U-235 fission.

In [ ]:
# Generate ENDF-like data (from Tutorial 1)
def simulate_endf_data(isotope='U-235', reaction='fission'):
    """Simulate ENDF evaluation (slightly different from TENDL)"""
    energy = np.logspace(-2, 7, 10000)
    
    if isotope == 'U-235' and reaction == 'fission':
        thermal = 584.4 * np.sqrt(0.0253 / np.maximum(energy, 1e-5))
        resonance = 50 + 30 * np.sin(np.log(np.maximum(energy, 1)))
        fast = 2.0 + 0.5 * np.log10(np.maximum(energy, 1e5) / 1e5)
        xs = np.where(energy < 1, thermal, np.where(energy < 1e4, resonance, fast))
    else:
        xs = np.ones_like(energy)
    
    return energy, xs

energy_endf, xs_fission_endf = simulate_endf_data('U-235', 'fission')

# Plot comparison
plt.figure(figsize=(14, 8))

plt.loglog(energy_endf, xs_fission_endf, label='ENDF/B-VIII.0', 
           linewidth=2.5, color='blue', alpha=0.7)
plt.loglog(energy_tendl, xs_fission_tendl, label='TENDL-2021', 
           linewidth=2.5, color='red', linestyle='--', alpha=0.7)

plt.xlabel('Neutron Energy (eV)', fontsize=14)
plt.ylabel('Fission Cross-section (barns)', fontsize=14)
plt.title('U-235 Fission Cross-Section: ENDF vs TENDL Comparison', fontsize=16)
plt.legend(fontsize=13)
plt.grid(True, alpha=0.3, which='both')
plt.xlim(1e-2, 1e7)
plt.tight_layout()
plt.savefig(data_dir / 'endf_vs_tendl_comparison.png', dpi=150)
plt.show()

## 6. Ratio Plot: Quantifying Differences

A ratio plot helps visualize the relative differences between evaluations.

In [ ]:
# Calculate ratio
ratio = xs_fission_tendl / xs_fission_endf

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), 
                               gridspec_kw={'height_ratios': [3, 1]})

# Top panel: Cross-sections
ax1.loglog(energy_endf, xs_fission_endf, label='ENDF/B-VIII.0', 
           linewidth=2.5, color='blue', alpha=0.7)
ax1.loglog(energy_tendl, xs_fission_tendl, label='TENDL-2021', 
           linewidth=2.5, color='red', linestyle='--', alpha=0.7)
ax1.set_ylabel('Fission Cross-section (barns)', fontsize=14)
ax1.set_title('U-235 Fission: ENDF vs TENDL with Ratio', fontsize=16)
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3, which='both')
ax1.set_xlim(1e-2, 1e7)

# Bottom panel: Ratio
ax2.semilogx(energy_tendl, ratio, linewidth=2, color='green')
ax2.axhline(1.0, color='black', linestyle='-', linewidth=1.5, alpha=0.5)
ax2.axhline(1.05, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax2.axhline(0.95, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Neutron Energy (eV)', fontsize=14)
ax2.set_ylabel('Ratio\n(TENDL/ENDF)', fontsize=12)
ax2.set_ylim(0.85, 1.15)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1e-2, 1e7)

plt.tight_layout()
plt.savefig(data_dir / 'endf_tendl_ratio.png', dpi=150)
plt.show()

# Statistics
print("\nRatio Statistics (TENDL/ENDF):")
print(f"  Mean: {np.mean(ratio):.4f}")
print(f"  Median: {np.median(ratio):.4f}")
print(f"  Std Dev: {np.std(ratio):.4f}")
print(f"  Min: {np.min(ratio):.4f}")
print(f"  Max: {np.max(ratio):.4f}")

## 7. Exploring Exotic Isotopes

One of TENDL's strengths is coverage of isotopes not in ENDF. Let's look at an example with a medical isotope.

In [ ]:
# Example: Mo-99 (used for medical Tc-99m production)
def simulate_mo99_data():
    """Simulate Mo-99 (n,gamma) cross-section from TENDL"""
    energy = np.logspace(-2, 7, 10000)
    
    # Typical (n,γ) behavior
    thermal = 0.5 * np.sqrt(0.0253 / np.maximum(energy, 1e-5))
    resonance = 0.3 + 0.2 * np.sin(2 * np.log(np.maximum(energy, 1)))
    fast = 0.01 + 0.005 * np.log10(np.maximum(energy, 1e5) / 1e5)
    
    xs = np.where(energy < 10, thermal, np.where(energy < 1e4, resonance, fast))
    return energy, xs

energy_mo99, xs_mo99_ng = simulate_mo99_data()

plt.figure(figsize=(14, 8))
plt.loglog(energy_mo99, xs_mo99_ng, linewidth=2.5, color='purple')
plt.xlabel('Neutron Energy (eV)', fontsize=14)
plt.ylabel('(n,γ) Cross-section (barns)', fontsize=14)
plt.title('Mo-99(n,γ)Mo-100 Cross-Section from TENDL-2021\n(Medical Isotope Production)', fontsize=16)
plt.grid(True, alpha=0.3, which='both')
plt.xlim(1e-2, 1e7)
plt.tight_layout()
plt.show()

print("Mo-99 is important for:")
print("  - Medical imaging (via Tc-99m decay product)")
print("  - This data may only be available in TENDL, not ENDF")
print(f"  - Thermal cross-section: {xs_mo99_ng[0]:.3f} barns")

## 8. TENDL Uncertainties

TENDL provides uncertainty data (covariance matrices) which are valuable for uncertainty quantification.

In [ ]:
# Simulate TENDL uncertainty bands (typically 5-15% for TALYS calculations)
uncertainty_thermal = 0.05  # 5% in thermal region
uncertainty_resonance = 0.10  # 10% in resonance region
uncertainty_fast = 0.08  # 8% in fast region

# Create uncertainty bands
unc = np.where(energy_tendl < 1, uncertainty_thermal,
               np.where(energy_tendl < 1e4, uncertainty_resonance, uncertainty_fast))

xs_upper = xs_fission_tendl * (1 + unc)
xs_lower = xs_fission_tendl * (1 - unc)

plt.figure(figsize=(14, 8))

# Plot central value
plt.loglog(energy_tendl, xs_fission_tendl, linewidth=2.5, 
           color='red', label='TENDL-2021 (central value)')

# Plot uncertainty band
plt.fill_between(energy_tendl, xs_lower, xs_upper, 
                 alpha=0.3, color='red', label='TENDL uncertainty (±1σ)')

# Add ENDF for comparison
plt.loglog(energy_endf, xs_fission_endf, linewidth=2, 
           color='blue', linestyle='--', alpha=0.7, label='ENDF/B-VIII.0')

plt.xlabel('Neutron Energy (eV)', fontsize=14)
plt.ylabel('Fission Cross-section (barns)', fontsize=14)
plt.title('U-235 Fission with TENDL Uncertainties', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3, which='both')
plt.xlim(1e-2, 1e7)
plt.tight_layout()
plt.savefig(data_dir / 'tendl_uncertainties.png', dpi=150)
plt.show()

## 9. Extracting Data for Machine Learning

Let's create a dataset combining TENDL data with uncertainties.

In [ ]:
# Create comprehensive dataset
df_tendl = pd.DataFrame({
    'Energy_eV': energy_tendl,
    'TENDL_Fission_barns': xs_fission_tendl,
    'TENDL_Capture_barns': xs_capture_tendl,
    'ENDF_Fission_barns': xs_fission_endf,
    'TENDL_Uncertainty': unc,
    'TENDL_Upper_barns': xs_upper,
    'TENDL_Lower_barns': xs_lower,
    'Ratio_TENDL_ENDF': xs_fission_tendl / xs_fission_endf
})

# Add some derived features for ML
df_tendl['Log10_Energy'] = np.log10(df_tendl['Energy_eV'])
df_tendl['Energy_Region'] = pd.cut(df_tendl['Energy_eV'], 
                                     bins=[0, 1, 1e4, 1e7],
                                     labels=['Thermal', 'Resonance', 'Fast'])

# Save to CSV
csv_file = data_dir / 'tendl_u235_comparison.csv'
df_tendl.to_csv(csv_file, index=False)

print(f"Data saved to: {csv_file}")
print(f"\nDataset shape: {df_tendl.shape}")
print("\nFirst few rows:")
print(df_tendl.head(10))

print("\nSummary by energy region:")
print(df_tendl.groupby('Energy_Region')['Ratio_TENDL_ENDF'].describe())

## 10. Real TENDL Data Access

### How to Download Real TENDL Data:

1. **Visit TENDL website**: https://tendl.web.psi.ch/

2. **Select version**: TENDL-2021 (or latest)

3. **Navigate to neutron data**: tendl_2021/neutron_file/

4. **Download specific isotope**: e.g., `n-U-235.tendl`

5. **Process with OpenMC**:
```python
import openmc.data
u235_tendl = openmc.data.IncidentNeutron.from_endf('n-U-235.tendl')

# Extract cross-section
energy = np.logspace(0, 7, 10000)
xs_fission = u235_tendl[18].xs['0K'](energy)  # MT=18 for fission
```

### Note on File Size:
- Individual TENDL files: 1-10 MB
- Full library: Several GB
- Download only what you need!

## 11. Key Takeaways

1. **TENDL provides extensive coverage** - Many more isotopes than ENDF
2. **Automated evaluation** - Consistent but may lack expert refinement
3. **Same format as ENDF** - Easy to process with same tools
4. **Includes uncertainties** - Valuable for ML uncertainty quantification
5. **Regularly updated** - New releases every 1-2 years
6. **Complementary to ENDF** - Use both for comprehensive analysis

### When to Use TENDL:
- Working with exotic/rare isotopes
- Need uncertainty data
- Medical isotope applications
- Activation calculations
- Quick evaluations for new isotopes

### When to Use ENDF:
- Critical reactor calculations
- Major fissile isotopes (U-235, Pu-239)
- Need highest accuracy
- Regulatory applications

## Next Steps

- Download real TENDL files for your isotopes
- Compare multiple evaluations (ENDF, TENDL, JEFF, JENDL)
- Use uncertainty data for ML model training
- Investigate discrepancies between libraries

## Resources

- TENDL Website: https://tendl.web.psi.ch/
- TALYS Code: https://tendl.web.psi.ch/tendl_2021/talys.html
- OpenMC Documentation: https://docs.openmc.org/
- Nuclear Data Evaluation: https://www-nds.iaea.org/ngatlas/